In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# Contoh data
data = {
    'luas_tanah': [120, 90, 150, 80, 200, 110, 130, 170, 95, 140,
                   160, 100, 180, 75, 220, 115, 135, 155, 85, 145],
    'jumlah_kamar': [4, 2, 5, 2, 6, 3, 4, 5, 3, 4,
                     5, 3, 6, 2, 7, 3, 4, 5, 2, 4],
    'kondisi': ['baik', 'sedang', 'baik', 'kurang', 'baik',
                'sedang', 'baik', 'baik', 'sedang', 'kurang',
                'baik', 'sedang', 'baik', 'kurang', 'baik',
                'sedang', 'sedang', 'baik', 'kurang', 'baik'],
    'harga': [1500, 900, 1800, 750, 2500, 1200, 1600, 2000, 1000, 1350,
              2100, 1050, 2300, 700, 2800, 1250, 1400, 1950, 850, 1700]
}
df = pd.DataFrame(data)
print(df.head())

   luas_tanah  jumlah_kamar kondisi  harga
0         120             4    baik   1500
1          90             2  sedang    900
2         150             5    baik   1800
3          80             2  kurang    750
4         200             6    baik   2500


In [30]:
# 1. Split data
X = df[['luas_tanah', 'jumlah_kamar', 'kondisi']]
y = df['harga']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [31]:
# 2. Preprocessing fitur (X)
scaler_luas = StandardScaler()
scaler_kamar = StandardScaler()

X_train_luas_scaled = scaler_luas.fit_transform(X_train[['luas_tanah']])
X_train_kamar_scaled = scaler_kamar.fit_transform(X_train[['jumlah_kamar']])
X_test_luas_scaled = scaler_luas.transform(X_test[['luas_tanah']])
X_test_kamar_scaled = scaler_kamar.transform(X_test[['jumlah_kamar']])

# --- Encoding fitur kategorikal dengan get_dummies ---
kondisi_dummies_train = pd.get_dummies(X_train['kondisi'], prefix='kondisi')
kondisi_dummies_test = pd.get_dummies(X_test['kondisi'], prefix='kondisi')

# Sinkronisasi kolom: pastikan kolom test sama persis dengan train
kondisi_dummies_test = kondisi_dummies_test.reindex(
    columns=kondisi_dummies_train.columns, fill_value=0
)

# --- Gabungkan semua fitur yang sudah diproses ---
X_train_final = np.column_stack([
    X_train_luas_scaled,
    X_train_kamar_scaled,
    kondisi_dummies_train.values
])

X_test_final = np.column_stack([
    X_test_luas_scaled,
    X_test_kamar_scaled,
    kondisi_dummies_test.values
])

print("X_train_final :")
print(X_train_final)
print("X_test_final :")
print(X_test_final)

X_train_final :
[[-0.9740652  -0.71902339  0.          0.          1.        ]
 [-0.61715581 -0.71902339  0.          0.          1.        ]
 [-0.8550954  -0.71902339  0.          0.          1.        ]
 [-1.33097458 -1.39575128  0.          1.          0.        ]
 [-1.21200479 -1.39575128  0.          1.          0.        ]
 [-0.02230684 -0.04229549  0.          0.          1.        ]
 [-1.44994438 -1.39575128  0.          1.          0.        ]
 [ 0.33460255  0.6344324   1.          0.          0.        ]
 [ 0.09666296 -0.04229549  0.          1.          0.        ]
 [ 0.21563275 -0.04229549  1.          0.          0.        ]
 [ 1.5243005   1.3111603   1.          0.          0.        ]
 [ 1.04842132  1.3111603   1.          0.          0.        ]
 [ 0.81048173  0.6344324   1.          0.          0.        ]
 [ 0.57254214  0.6344324   1.          0.          0.        ]
 [ 2.00017968  1.98788819  1.          0.          0.        ]
 [-0.14127663 -0.04229549  1.          

In [32]:
# 3. Preprocessing target (y)
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# 4. Training model
model = LinearRegression()
model.fit(X_train_final, y_train_scaled)

# 5. Prediksi dan inverse transform
y_pred_scaled = model.predict(X_test_final)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

# 6. Evaluasi
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")
print(f"MAE: Rp {mean_absolute_error(y_test, y_pred):.2f} Juta")

# Tampilkan perbandingan
hasil = pd.DataFrame({
    'Harga Aktual': y_test.values,
    'Harga Prediksi': y_pred
})
print("\nPerbandingan Prediksi:")
print(hasil)

R2 Score: 0.99
MAE: Rp 34.09 Juta

Perbandingan Prediksi:
   Harga Aktual  Harga Prediksi
0          1500     1527.656120
1          1950     1943.917602
2          1250     1185.587362
3           900      861.800594


Logistic regression

In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Contoh data
data = {
    'pendapatan': [8, 3, 12, 2, 15, 5, 7, 10, 4, 6,
                   9, 11, 14, 3, 8, 13, 4, 6, 16, 2],
    'usia': [35, 28, 45, 22, 50, 30, 33, 40, 25, 38,
             42, 37, 48, 26, 31, 46, 24, 34, 52, 21],
    'status_kerja': ['tetap', 'kontrak', 'tetap', 'kontrak', 'tetap',
                     'kontrak', 'tetap', 'tetap', 'kontrak', 'tetap',
                     'tetap', 'kontrak', 'tetap', 'kontrak', 'tetap',
                     'tetap', 'kontrak', 'tetap', 'tetap', 'kontrak'],
    'kelayakan': ['Layak', 'Tidak Layak', 'Layak', 'Tidak Layak', 'Layak',
                  'Tidak Layak', 'Layak', 'Layak', 'Tidak Layak', 'Layak',
                  'Layak', 'Tidak Layak', 'Layak', 'Tidak Layak', 'Layak',
                  'Layak', 'Tidak Layak', 'Layak', 'Layak', 'Tidak Layak']
}
df = pd.DataFrame(data)
print(df.head())

   pendapatan  usia status_kerja    kelayakan
0           8    35        tetap        Layak
1           3    28      kontrak  Tidak Layak
2          12    45        tetap        Layak
3           2    22      kontrak  Tidak Layak
4          15    50        tetap        Layak


In [34]:
# 1. Split data
X = df[['pendapatan', 'usia', 'status_kerja']]
y = df['kelayakan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
# 2. Encoding target (kelayakan) dengan LabelEncoder
label_encoder_y = LabelEncoder()
y_train_binary = label_encoder_y.fit_transform(y_train)  # 0=Tidak Layak, 1=Layak
y_test_binary = label_encoder_y.transform(y_test)

# 3. Encoding fitur kategorikal (status_kerja) dengan LabelEncoder
label_encoder_status = LabelEncoder()
X_train['status_kerja'] = label_encoder_status.fit_transform(X_train['status_kerja'])  # 0=kontrak, 1=tetap
X_test['status_kerja'] = label_encoder_status.transform(X_test['status_kerja'])

# 4. Scaling semua fitur
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train)
X_test_final = scaler.transform(X_test)

print("X_train_final :")
print(X_train_final)
print("X_test_final :")
print(X_test_final)

X_train_final :
[[-0.89164514 -1.03114068 -1.13389342]
 [-0.66873386 -0.52814523 -1.13389342]
 [ 0.66873386  0.17604841 -1.13389342]
 [-1.33746771 -1.33293796 -1.13389342]
 [ 1.78329028  1.68503477  0.8819171 ]
 [-0.89164514 -1.13173977 -1.13389342]
 [-1.11455643 -0.93054159 -1.13389342]
 [ 0.89164514  0.98084114  0.8819171 ]
 [-0.44582257  0.2766475   0.8819171 ]
 [-1.33746771 -1.43353705 -1.13389342]
 [ 1.560379    1.48383659  0.8819171 ]
 [ 1.33746771  1.28263841  0.8819171 ]
 [ 0.44582257  0.47784568  0.8819171 ]
 [ 0.22291129  0.67904386  0.8819171 ]
 [ 0.         -0.42754614  0.8819171 ]
 [-0.22291129 -0.22634795  0.8819171 ]]
X_test_final :
[[ 0.         -0.02514977  0.8819171 ]
 [-0.44582257 -0.12574886  0.8819171 ]
 [ 1.11455643  1.08144023  0.8819171 ]
 [-1.11455643 -0.72934341 -1.13389342]]


In [36]:
# 5. Training model
model = LogisticRegression()
model.fit(X_train_final, y_train_binary)

# 6. Prediksi
y_pred = model.predict(X_test_final)

# 7. Evaluasi
print("\n=== Hasil Evaluasi ===")
print(f"Akurasi: {accuracy_score(y_test_binary, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test_binary, y_pred,
                            target_names=label_encoder_y.classes_))

# 8. Tampilkan data uji dengan prediksi
X_test_asli = df.iloc[X_test.index]  # ambil data asli sebelum transformasi
hasil = pd.DataFrame({
    'Pendapatan': X_test_asli['pendapatan'].values,
    'Usia': X_test_asli['usia'].values,
    'Status Kerja': X_test_asli['status_kerja'].values,
    'Aktual': y_test.values,
    'Prediksi': label_encoder_y.inverse_transform(y_pred)
})
print("\nPerbandingan Prediksi Data Uji:")
print(hasil)


=== Hasil Evaluasi ===
Akurasi: 1.00

Classification Report:
              precision    recall  f1-score   support

       Layak       1.00      1.00      1.00         3
 Tidak Layak       1.00      1.00      1.00         1

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4


Perbandingan Prediksi Data Uji:
   Pendapatan  Usia Status Kerja       Aktual     Prediksi
0           8    35        tetap        Layak        Layak
1           6    34        tetap        Layak        Layak
2          13    46        tetap        Layak        Layak
3           3    28      kontrak  Tidak Layak  Tidak Layak
